# WAN Distributed Colab Worker

Run the original WAN notebook's **Prepare Environment** setup first. This notebook only starts the VPS worker and does not replace or reimplement the generation code.

Set `VPS_URL` to the public VPS address, for example `http://YOUR_EC2_PUBLIC_IP:8000`.

In [ ]:
import os
import sys
from pathlib import Path

os.system("pip install -q httpx")
os.environ["VPS_URL"] = "http://54.169.43.89/:8000"
os.environ.setdefault("WORKER_NAME", "colab-t4")
os.environ.setdefault("POLL_INTERVAL_SECONDS", "15")

COMFY_INPUT = Path("/content/ComfyUI/input")
COMFY_OUTPUT = Path("/content/ComfyUI/output")
COMFY_INPUT.mkdir(parents=True, exist_ok=True)
COMFY_OUTPUT.mkdir(parents=True, exist_ok=True)

print("VPS_URL:", os.environ["VPS_URL"])
print("Worker name:", os.environ["WORKER_NAME"])


In [ ]:
import gc
import sys
import __main__

required = ["generate_video", "torch", "output_path"]
missing = [name for name in required if name not in __main__.__dict__ and name != "output_path"]
if missing:
    raise RuntimeError(
        "Run the WAN notebook Prepare Environment cell first. Missing: "
        + ", ".join(missing)
    )

torch = __main__.torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

__main__.setdefault("clear_output", lambda *args, **kwargs: None) if hasattr(__main__, "setdefault") else None
if "clear_output" not in __main__.__dict__:
    __main__.__dict__["clear_output"] = lambda *args, **kwargs: None
if "display" not in __main__.__dict__:
    __main__.__dict__["display"] = lambda *args, **kwargs: None

COMFY_INPUT.mkdir(parents=True, exist_ok=True)
COMFY_OUTPUT.mkdir(parents=True, exist_ok=True)
print("Input:", COMFY_INPUT)
print("Output:", COMFY_OUTPUT)


## Install the worker package

Upload `worker/__init__.py`, `worker/config.py`, `worker/wan_adapter.py`, and `worker/colab_worker.py` from this project into `/content/wan_worker/`, or copy that directory into `/content/wan_worker/` from Drive before running the next cell.

In [ ]:
from pathlib import Path

worker_root = Path("/content/wan_worker")
required_files = ["__init__.py", "config.py", "wan_adapter.py", "colab_worker.py"]
missing_files = [name for name in required_files if not (worker_root / name).is_file()]
if missing_files:
    raise RuntimeError("Upload/copy worker files into /content/wan_worker/: " + ", ".join(missing_files))

sys.path.insert(0, "/content")
from wan_worker import config, colab_worker

print("Worker package ready:", worker_root)


In [ ]:
import httpx

if not config.VPS_URL or "YOUR_EC2" in config.VPS_URL:
    raise RuntimeError("Set VPS_URL to the public URL of the Phase 1 server")

health = httpx.get(config.VPS_URL + "/health", timeout=15.0)
print("VPS health:", health.status_code, health.text)
health.raise_for_status()

print("Starting worker loop. Press the Colab stop button to interrupt.")
colab_worker.run_forever()
